# 06 Training

This notebook trains the final production model using the selected features and the tuned parameters written back into `params.yaml`.

Outputs:
- final model artifact
- metrics JSON
- test prediction CSV


## Research Paper Alignment

Training stays separate from the research benchmark on purpose. The deployable model is optimized for reproducibility and serving, while the research stage explores broader hybrid and regime-sensitive ideas.


In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import yaml

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "params.yaml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["figure.dpi"] = 130

ASSET_DIR = PROJECT_ROOT / "presentation" / "figures"
ASSET_DIR.mkdir(parents=True, exist_ok=True)

def read_json(path: str):
    return json.loads(Path(path).read_text(encoding="utf-8"))

params = yaml.safe_load(Path("params.yaml").read_text(encoding="utf-8"))


In [ ]:
params["training"]


In [ ]:
subprocess.run([sys.executable, "dvc_pipeline/src/train_model.py"], check=True)


In [ ]:
metrics = read_json(params["training"]["metrics_path"])
pred_df = pd.read_csv(params["training"]["predictions_path"])
pred_df["Date"] = pd.to_datetime(pred_df["Date"])

metrics, pred_df.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

trace_df = pred_df.tail(100)
sns.lineplot(data=trace_df, x="Date", y="actual", ax=axes[0], label="Actual", linewidth=2.2, color="black")
sns.lineplot(data=trace_df, x="Date", y="predicted", ax=axes[0], label="Predicted", linewidth=1.7)
axes[0].set_title("Production Model Fit on Test Window")
axes[0].set_ylabel("Price")

residuals = pred_df["actual"] - pred_df["predicted"]
sns.histplot(residuals, bins=25, kde=True, ax=axes[1], color="#4c51bf")
axes[1].set_title("Residual Distribution")
axes[1].set_xlabel("Actual - Predicted")

fig.tight_layout()
fig.savefig(ASSET_DIR / "10_production_fit.png", bbox_inches="tight")
plt.show()
